# Sentiment Classification Using BERT

Your LSTM in `sentimentAnalysis.ipynb` learned English **from scratch**, using nothing but the
movie reviews. That is like teaching someone the language *and* the job at the same time.

**BERT** (Bidirectional Encoder Representations from Transformers) arrives already able to
read English. Google trained it on all of Wikipedia plus thousands of books, by hiding
random words and making it guess them back. To get good at that, it had to learn grammar,
meaning and context.

Two things make it a better reader than the LSTM:

- **It reads in both directions at once** ("bidirectional"). The LSTM reads left to right
  and by word 180 has mostly forgotten word 3. BERT looks at every word together and works
  out which words matter to which. That is how it can link "but" to both halves of
  *"I expected it to be awful, but it turned out great"*.
- **It already knows English.** We only have to teach it the job (positive or negative).
  This is **fine-tuning**, the same idea as reusing ResNet18 for cats and dogs.

What changes from the LSTM code:

- **The tokenizer** (BERT's own text-to-numbers converter) replaces `clean_text`, the
  `vocab` dictionary and `encode`. It uses the exact dictionary BERT was trained with.
- **A tiny learning rate.** `2e-5` (0.00002) is fifty times smaller than the LSTM's `0.001`.
  BERT already knows English, and big nudges would wreck that knowledge.
- **It is big.** 110 million numbers against the LSTM's roughly 800,000. So we train on a
  slice of the reviews for just one epoch, to keep it to a few minutes.

The first run downloads BERT (about 440MB). After that it is cached on your machine.

In [ ]:
#step 1 - tools and data

import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"using device-{device}")

dataframe = pd.read_csv('IMDB_Dataset.csv')

# exactly the same split as the LSTM notebook, so the scores are comparable
training_df = dataframe.sample(frac=0.7, random_state=42)
test_df = dataframe.drop(training_df.index)

print(f"{len(training_df)} training reviews, {len(test_df)} test reviews")

In [ ]:
#step 2 - turn the reviews into numbers with BERT's own tokenizer

MODEL_NAME = "bert-base-uncased"   # plain BERT: knows English, has never judged a review
MAX_LEN = 256                      # how many word-pieces of each review to read (BERT's limit is 512)
HOW_MANY = 5000                    # training reviews to use. more = better score, but slower

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# a slice of each, to keep it quick
train_part = training_df.head(HOW_MANY)
test_part = test_df.head(2000)


def prepare(df):
    """Turn a table of reviews into number-blocks BERT can read."""
    packed = tokenizer(
        df["review"].tolist(),
        truncation=True,          # cut reviews longer than MAX_LEN
        padding="max_length",     # pad short ones with blanks so every review is the same length
        max_length=MAX_LEN,
        return_tensors="pt",      # hand back PyTorch tensors
    )
    labels = torch.tensor(df["sentiment"].map({"negative": 0, "positive": 1}).values)
    # attention_mask: 1 = real word, 0 = blank filler, so BERT knows to ignore the padding
    return TensorDataset(packed["input_ids"], packed["attention_mask"], labels)


train_dataloader = DataLoader(prepare(train_part), batch_size=16, shuffle=True)
test_dataloader = DataLoader(prepare(test_part), batch_size=32)

# peek at how the tokenizer chops up a sentence
print(tokenizer.tokenize("I expected it to be awful, but honestly it turned out great."))
print(f"{len(train_part)} training reviews, {len(test_part)} test reviews, ready")

In [ ]:
#step 3 - fine-tune it (the same training loop as the LSTM)

epochs = 1   # one pass is usually enough for BERT

# num_labels=2 bolts a fresh 2-way decision layer (negative, positive) onto the end of BERT.
# it prints a LOAD REPORT: "classifier ... MISSING" means that layer is new (the part we train),
# "cls... UNEXPECTED" is BERT's old word-guessing layer being thrown away. both are expected
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(device)

loss_fun = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)    # tiny steps: it already knows English


def check_score():
    """Grade the model on the test reviews, give back the % it got right."""
    model.eval()                                  # exam mode: dropout switched off
    correct = 0
    total = 0
    with torch.no_grad():                         # just guessing, no learning
        for ids, mask, labels in test_dataloader:
            ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)
            guess = model(ids, attention_mask=mask).logits    # .logits holds the 2 raw scores
            answer = guess.argmax(dim=1)                      # whichever of the 2 scores is higher
            correct += (answer == labels).sum().item()
            total += len(labels)
    model.train()                                 # back to learning mode
    return correct / total * 100


model.train()
for epoch in range(epochs):
    total_loss = 0

    for ids, mask, labels in train_dataloader:
        ids, mask, labels = ids.to(device), mask.to(device), labels.to(device)

        optimizer.zero_grad()                              # wipe the last batch's corrections
        guess = model(ids, attention_mask=mask).logits     # what does it think these are?
        loss = loss_fun(guess, labels)                     # how wrong was it?
        loss.backward()                                    # work out which way to nudge things
        optimizer.step()                                   # nudge

        total_loss += loss.item()

    print(f"Epoch {epoch+1}: avg loss {total_loss / len(train_dataloader):.4f}  test {check_score():.1f}%")

# save the model and its tokenizer together in one folder
model.save_pretrained("sentiment_bert")
tokenizer.save_pretrained("sentiment_bert")
print("saved to sentiment_bert/")

In [ ]:
#step 4 - try it on your own reviews

def how_does_it_feel(review):
    """Say whether one review is positive or negative, and how sure the model is."""
    packed = tokenizer(review, truncation=True, max_length=MAX_LEN, return_tensors="pt")
    ids = packed["input_ids"].to(device)
    mask = packed["attention_mask"].to(device)

    model.eval()
    with torch.no_grad():
        scores = model(ids, attention_mask=mask).logits   # 2 raw scores: negative, positive
        chances = torch.softmax(scores, dim=1)[0]         # turn them into percentages adding up to 100

    if chances[1] > chances[0]:
        return "positive", chances[1].item() * 100
    return "negative", chances[0].item() * 100


# the same reviews as the LSTM notebook, so you can compare the two readers. change or add your own
for review in [
    "This movie was wonderful. Brilliant acting and I loved every minute of it.",
    "What a waste of two hours. Boring plot, terrible acting, I want my money back.",
    "I expected it to be awful, but honestly it turned out to be great.",
    "It was not good.",
]:
    mood, sureness = how_does_it_feel(review)
    print(f"{mood} ({sureness:.1f}% sure)  <-  {review}")